# US WARN Act layoff notices: a starter analysis

[WARN Index](https://warnindex.com) collects Worker Adjustment and Retraining Notification
(WARN) Act notices from US state labor agencies every day and normalizes them into one
table. This notebook loads the weekly snapshot and looks at:

1. **Coverage**: which states the data covers, and since when
2. **Trends**: monthly notices and affected workers
3. **Where**: states and worksites
4. **Who**: industries and the largest recent notices

Data: CC BY 4.0, please credit **WARN Index** (https://warnindex.com). Live data, maps and
an API: https://warnindex.com/api-docs

In [ ]:
import glob

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 60)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False})

# On Kaggle the dataset is mounted under /kaggle/input; elsewhere read the
# same snapshot from Hugging Face.
paths = glob.glob("/kaggle/input/**/warn_dataset.parquet", recursive=True)
src = paths[0] if paths else (
    "https://huggingface.co/datasets/warn-index/us-warn-layoffs/resolve/main/warn_dataset.parquet"
)
df = pd.read_parquet(src)
df["notice_date"] = pd.to_datetime(df["notice_date"], errors="coerce")
print(f"{len(df):,} notices from {df['state'].nunique()} states, "
      f"{df['notice_date'].min():%Y-%m-%d} to {df['notice_date'].max():%Y-%m-%d}")
df.head()

Each row is one notice. `layoff_count` is the number of affected workers (empty when the
state does not publish one), `closure_category` is `Closure`, `Layoff` or `Non-WARN`, and
`detail_url` links to the notice on WARN Index. Full column descriptions are in the
dataset's Data Card.

In [ ]:
df.info()

## 1. Coverage: read trends with care

States publish their WARN history back to very different years, so the number of states in
the data grows over time. **A rising yearly total before ~2015 mostly reflects more states,
not more layoffs.** For trends, compare a fixed set of states (below) or a single state.

In [ ]:
by_year = df.groupby(df["notice_date"].dt.year).agg(
    notices=("notice_id", "size"), states=("state", "nunique"))
by_year.index = by_year.index.astype(int)

fig, ax = plt.subplots()
ax.bar(by_year.index, by_year["states"], color="#9aa5b1")
ax.set(title="States with at least one notice, by year", xlabel="", ylabel="states")
plt.show()

## 2. Trends in a fixed panel of states

To compare years fairly, keep only states that have notices in every year from 2012 on.
2020 stands out: the COVID-19 shutdowns produced the largest wave of WARN notices on record.

In [ ]:
recent = df[df["notice_date"] >= "2012-01-01"]
years_per_state = recent.groupby("state")["notice_date"].apply(lambda s: s.dt.year.nunique())
full_years = recent["notice_date"].dt.year.nunique() - 1  # the current year may be partial
panel = sorted(years_per_state[years_per_state >= full_years].index)
print(f"{len(panel)} panel states: {', '.join(panel)}")

monthly = (recent[recent["state"].isin(panel)]
           .set_index("notice_date")
           .resample("MS")
           .agg({"notice_id": "size", "layoff_count": "sum"})
           .rename(columns={"notice_id": "notices", "layoff_count": "workers"}))
monthly = monthly.iloc[:-1]  # drop the partial current month

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(monthly.index, monthly["notices"], color="#da2f25")
axes[0].set(title="WARN notices per month (panel states)", ylabel="notices")
axes[1].plot(monthly.index, monthly["workers"], color="#334e68")
axes[1].set(title="Workers affected per month (panel states)", ylabel="workers")
plt.tight_layout()
plt.show()

## 3. Where: states and worksites in the last 12 months

In [ ]:
cutoff = df["notice_date"].max() - pd.DateOffset(months=12)
last12 = df[df["notice_date"] > cutoff]

top_states = (last12.groupby("state")
              .agg(notices=("notice_id", "size"), workers=("layoff_count", "sum"))
              .sort_values("workers", ascending=False)
              .head(15))

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(top_states.index[::-1], top_states["workers"][::-1], color="#da2f25")
ax.set(title=f"Workers affected by state, last 12 months (to {df['notice_date'].max():%Y-%m-%d})",
       xlabel="workers")
plt.show()
top_states

In [ ]:
pts = last12.dropna(subset=["lat", "lon"])
pts = pts[pts["lon"].between(-125, -66) & pts["lat"].between(24, 50)]  # lower 48

fig, ax = plt.subplots(figsize=(11, 6.5))
sizes = (pts["layoff_count"].fillna(10).clip(upper=2000) / 8).astype(float)
ax.scatter(pts["lon"], pts["lat"], s=sizes, alpha=0.35, color="#da2f25", linewidths=0)
ax.set(title="WARN notice worksites, last 12 months (size = workers)",
       xlabel="longitude", ylabel="latitude")
ax.set_aspect(1.25)
plt.show()

## 4. Who: industries and the largest recent notices

Industries come from WARN Index's employer matching (NAICS). Notices whose employer could
not be matched have no industry.

In [ ]:
industries = (last12.dropna(subset=["naics_desc"])
              .groupby("naics_desc")
              .agg(notices=("notice_id", "size"), workers=("layoff_count", "sum"))
              .sort_values("workers", ascending=False)
              .head(15))
share = last12["naics_desc"].notna().mean()
print(f"{share:.0%} of last-12-month notices have an industry")
industries

In [ ]:
cols = ["notice_date", "state", "employer", "city", "layoff_count", "closure_category",
        "detail_url"]
(df[df["notice_date"] > df["notice_date"].max() - pd.Timedelta(days=90)]
 .sort_values("layoff_count", ascending=False)[cols]
 .head(15))

## Next steps

- Join snapshots across dataset versions on `notice_id`, which is stable between versions.
- Live filings, state pages, alerts and a REST API: https://warnindex.com
- Cite: WARN Index (2026), *WARN Index: US WARN Act layoff notices dataset*, Zenodo,
  https://doi.org/10.5281/zenodo.23902958